# RQ2: TextVQA100 — native ApET and pure random

**Deferred optional input-only ablation. Run the new staged calibration first**
(`kaggle_rq2_staged_calibration.ipynb`); broader matched-schedule evaluation has priority.

Run all **100 existing screening questions**, including baseline-wrong cases.
At each **input budget 288 and 64**, compare native ApET, identical selection
without merging, and pure random at seeds 101/202/303. Full576 is the reference:
**1,100 primary predictions**, one model load, conditions run sequentially.
No decoder pruning, token interventions, or new teacher-forced target audits.

Use **Internet + T4 ×2**, the existing **rq1-ih-inputs-v1** private Dataset,
and the locked original environment. No new input upload or model-library change.
The six-case calibration has already passed; this is its broader evaluation.
All100 and the 94 outside the previously studied six are reported separately.
This is a screening/development cohort, not held-out/full TextVQA or a speed benchmark.

Push code first. Set `RUN_TEXTVQA100 = True` below, then run cells in order.
Default False prevents an imported/background notebook from starting inference.
See `research/RQ2_TEXTVQA100.md`.


In [ ]:
import json
import os
import re
import shutil
import subprocess
import sys
import zipfile
from datetime import datetime, timezone
from pathlib import Path

RUN_TEXTVQA100 = False  # Set True to run all100 questions at both input budgets.
REPO_URL = "https://github.com/doguineyes/CSE590-ApET.git"
REPO_REF = "apet-analysis"
INPUT_ZIP = None  # Existing rq1-ih-inputs-v1 Dataset/ZIP; not the older rq1-inputs-v1.
CHECKPOINT_ARGS = []  # Optional existing original-format model and vision paths.

if not RUN_TEXTVQA100:
    raise RuntimeError("Set RUN_TEXTVQA100 = True, then run cells manually.")
print("Preparing TextVQA100 comparison code; model inference has not started.", flush=True)
repo = Path("/kaggle/working/CSE590-ApET")
if not repo.exists():
    subprocess.run(["git", "clone", REPO_URL, str(repo)], check=True, timeout=120)
subprocess.run(["git", "fetch", "origin"], cwd=repo, check=True, timeout=120)
subprocess.run(["git", "checkout", REPO_REF], cwd=repo, check=True, timeout=120)
if REPO_REF == "apet-analysis":
    subprocess.run(["git", "pull", "--ff-only"], cwd=repo, check=True, timeout=120)
subprocess.run(["git", "rev-parse", "HEAD"], cwd=repo, check=True, timeout=120)
if not (repo / "scripts/run_rq2_textvqa100.py").is_file():
    raise RuntimeError("Push the new TextVQA100 files first, then update this checkout.")
original_python = repo / "environments/original/.venv/bin/python"
rq2_textvqa_runs = []
print("Repository:", repo, flush=True)


## Attach the existing inputs and prepare the original runtime

The attached `rq1-ih-inputs-v1` Dataset already contains all100 TextVQA questions and images. Reuse the environment/cache in the same live session; separate Kaggle sessions do not share them.


In [ ]:
sys.path.insert(0, str(repo / "scripts"))
from rq1_input_ih import locate_inputs
inputs = locate_inputs(
    [Path("/kaggle/input"), Path("/kaggle/working")],
    Path("/kaggle/working/rq1-ih-inputs-v1"), explicit=INPUT_ZIP,
)
print("Accepted input directory:", inputs, flush=True)
setup_stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
setup_log = Path(f"/kaggle/working/rq2-textvqa100-setup-{setup_stamp}.log")
runtime_report = Path(f"/kaggle/working/rq2-textvqa100-runtime-{setup_stamp}.json")

def stream_command(command, stream, *, cwd=None):
    print("Running:", " ".join(map(str, command)), flush=True)
    stream.write("COMMAND " + repr(command) + "\n")
    stream.flush()
    process = subprocess.Popen(command, cwd=cwd, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, encoding="utf-8", bufsize=1)
    lines = []
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
            stream.write(line)
            stream.flush()
            lines.append(line)
        status = process.wait()
    except BaseException:
        process.terminate()
        try:
            process.wait(timeout=20)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        raise
    finally:
        process.stdout.close()
    stream.write(f"EXIT {status}\n")
    stream.flush()
    if status:
        raise RuntimeError(f"Command failed (exit {status}); preserve {stream.name} before changing anything.")
    return "".join(lines)

with setup_log.open("x", encoding="utf-8") as stream:
    stream_command([
        "bash", "bootstrap/setup_original_linux.sh", "--require-cuda", "--expected-gpus", "2",
        "--output", str(runtime_report),
    ], stream, cwd=repo)
print("Locked original runtime ready. Run the prerequisite/helper cell next.", flush=True)
print("Setup log:", setup_log)
print("Runtime report:", runtime_report)


## Run prerequisites before loading pretrained weights

Existing tiny native64/random tests plus the new cohort, paired-statistics, generation-only and export tests. Zero skips are required. A saved prerequisite proof binds the test result to the current source files and runtime.


In [ ]:
def verify_rq2_textvqa_prerequisites(*, refresh_runtime=False):
    globals()["rq2_textvqa_prerequisites_passed"] = False
    with setup_log.open("a", encoding="utf-8") as stream:
        if refresh_runtime:
            stream_command([str(original_python), "-X", "utf8", "-I", "scripts/check_original_env.py",
                "--require-cuda", "--expected-gpus", "2", "--output", str(runtime_report)], stream, cwd=repo)
        output = stream_command([str(original_python), "-X", "utf8", "-m", "unittest",
            "research.tests.test_rq2_native", "research.tests.test_rq2_random_protocol",
            "research.tests.test_rq2_random_native", "research.tests.test_rq2_random_runner",
            "research.tests.test_rq2_textvqa_protocol", "research.tests.test_rq2_textvqa_runner",
            "research.tests.test_rq2_textvqa_native", "-v"], stream, cwd=repo)
        count = re.search(r"Ran (\d+) tests?", output)
        if not count or int(count.group(1)) <= 26 or re.search(r"skipped\s*=", output):
            raise RuntimeError("Require all new and existing actual native tests with zero skips; preserve setup log.")
    command = [str(original_python), "-X", "utf8", "-I", "-c",
        "import json,sys; from pathlib import Path; sys.path.insert(0,str(Path.cwd()/'scripts')); "
        "from run_rq2_textvqa100 import prerequisite_signature,TESTS; "
        "print(json.dumps({'source_sha256':prerequisite_signature(),'test_modules':list(TESTS)}))"]
    signature = json.loads(subprocess.run(command, cwd=repo, capture_output=True, text=True,
                                          check=True, timeout=120).stdout)
    import hashlib
    proof = {"version": "rq2-textvqa100-random-v1", "status": "passed", "tests_run": int(count.group(1)),
        "skipped": 0, **signature,
        "runtime_report_sha256": hashlib.sha256(runtime_report.read_bytes()).hexdigest(),
        "setup_log_sha256": hashlib.sha256(setup_log.read_bytes()).hexdigest()}
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    globals()["textvqa_prerequisite_report"] = Path(f"/kaggle/working/rq2-textvqa100-prerequisites-{stamp}.json")
    textvqa_prerequisite_report.write_text(json.dumps(proof, indent=2) + "\n")
    globals()["rq2_textvqa_prerequisites_passed"] = True
    print("TextVQA100 prerequisites passed:", proof["tests_run"], "tests, zero skips.", flush=True)


def run_rq2_textvqa100():
    if not globals().get("rq2_textvqa_prerequisites_passed", False):
        raise RuntimeError("Run verify_rq2_textvqa_prerequisites() successfully before loading weights.")
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    folder = Path(f"/kaggle/working/rq2-textvqa100-random-v1-{stamp}")
    archive, log = folder.with_suffix(".zip"), folder.with_suffix(".log")
    globals().setdefault("rq2_textvqa_runs", []).append(folder)
    command = [str(original_python), "-X", "utf8", "-I", "scripts/run_rq2_textvqa100.py",
        "--inputs", str(inputs), "--output", str(folder), "--run-textvqa100",
        "--require-cuda", "--expected-gpus", "2", "--download-budget-gib", "20",
        "--setup-log", str(setup_log), "--runtime-report", str(runtime_report),
        "--prerequisite-report", str(textvqa_prerequisite_report), *CHECKPOINT_ARGS]
    print("Run directory:", folder, flush=True)
    print("All100 questions ×11 primary conditions, sequentially; one model load. Controls are additional.", flush=True)
    fallback_archive = False
    try:
        with log.open("x", encoding="utf-8") as stream:
            stream_command(command, stream, cwd=repo)
    finally:
        if not archive.exists():
            fallback_archive = True
            fallback = folder / "notebook-fallback"
            fallback.mkdir(parents=True, exist_ok=True)
            (fallback / "README.txt").write_text("Incomplete fallback: runner ZIP missing. No successful run established.\n")
            for path in [setup_log, runtime_report, textvqa_prerequisite_report, log]:
                if path.is_file():
                    shutil.copy2(path, fallback / path.name)
            with zipfile.ZipFile(archive, "x", zipfile.ZIP_DEFLATED) as bundle:
                for path in sorted(folder.rglob("*")):
                    if path.is_file():
                        bundle.write(path, path.relative_to(folder))
        print("Download:", archive, "and", log, flush=True)
        if (folder / "report.json").is_file():
            report = json.loads((folder / "report.json").read_text())
            print("Status:", report.get("status"), "Progress:", report.get("progress"), flush=True)
            print("Wall seconds:", report.get("wall_seconds"), flush=True)
            if report.get("error"):
                print("Error:", report["error"], flush=True)
    if fallback_archive:
        raise RuntimeError("Runner ZIP missing; retain the incomplete fallback ZIP/log.")
    report = json.loads((folder / "report.json").read_text())
    if report.get("version") != "rq2-textvqa100-random-v1" or report.get("status") != "passed" or not report.get("execution_passed"):
        raise RuntimeError("TextVQA100 execution did not pass; download ZIP/log for review.")
    for subset in ("all100", "outside_previous_six"):
        summary = report["summaries"][subset]
        print(subset, "questions:", summary["questions"], flush=True)
        for name, value in summary["conditions"].items():
            print(name, "mean consensus:", round(value["mean_consensus_score"], 4),
                  "full credit:", value["full_credit_count"], "APR:", value["APR"], flush=True)
    return folder


# Re-execution invalidates any earlier test pass; rerun prerequisites after updates.
globals()["rq2_textvqa_prerequisites_passed"] = False


In [ ]:
verify_rq2_textvqa_prerequisites()


## Run all100 questions

No correctness filter. Eleven primary conditions per question, executed one by one. Each question's raw answers and controls are saved as it progresses. Exact clone replay and full576 parity are controls; fresh native numerical variation is recorded separately. Full tensors are saved only for Brolio and Dominion.


In [ ]:
textvqa100_dir = run_rq2_textvqa100()


## Download before ending the session

Download the timestamped **ZIP and matching log**. The ZIP includes all predictions, scores, consensus/APR summaries, paired score intervals, a 100-case gallery, source/input identities, runtime and prerequisite evidence. Keep a failed/partial archive too. Setup/runtime/prerequisite files also appear separately below.


In [ ]:
from IPython.display import FileLink, display
paths = [setup_log, runtime_report, textvqa_prerequisite_report]
for folder in rq2_textvqa_runs:
    paths.extend([folder.with_suffix(".zip"), folder.with_suffix(".log")])
for path in paths:
    if path.exists():
        display(FileLink(str(path)))
